# Inspect one raw battery JSON file

**Purpose (Week 1):** Open a single cell file with `json.load`, see what fields exist, and make a few plots — **without** processing all 140 files.

**You will learn:**
- How Python reads a `.json` file into a nested dictionary
- The difference between **per-cycle summary** (small table) and **interpolated curves** (large arrays)
- Which columns you can export later for machine learning

## 0. Setup paths

This notebook lives in `notebooks/`. Raw data is in `data/raw/`. We move up one folder so paths work.

In [ ]:
import os
from pathlib import Path

# If your kernel starts in notebooks/, go to project root
if Path.cwd().name == 'notebooks':
    os.chdir('..')

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
print('Project root:', PROJECT_ROOT)
print('Raw data dir:', RAW_DIR)

## 1. Pick one file and load it with `json.load`

`json.load(file)` reads the whole file and returns a **Python dictionary** (`dict`).
- Keys are strings (e.g. `'barcode'`, `'summary'`)
- Values can be numbers, lists, or more nested dicts

We only load **one** file here so this notebook stays fast (seconds, not minutes).

In [ ]:
import json
import glob

files = sorted(glob.glob(str(RAW_DIR / 'FastCharge*.json')))
print(f'Found {len(files)} JSON files in data/raw/')

sample_path = files[0]  # change index to try another cell
print('Inspecting:', sample_path)

with open(sample_path, 'r', encoding='utf-8') as f:
    cell = json.load(f)

print('Loaded successfully. Top-level type:', type(cell))

## 2. Top-level keys — what is inside one cell file?

Think of this as the "table of contents" for one battery.

In [ ]:
print('Top-level keys:')
for key in cell.keys():
    value = cell[key]
    if isinstance(value, dict):
        detail = f'dict with {len(value)} keys'
    elif isinstance(value, list):
        detail = f'list, length {len(value)}'
    else:
        detail = repr(value)[:80]
    print(f'  {key:25} -> {detail}')

In [ ]:
print('Cell barcode (ID):', cell.get('barcode'))
print('Channel:', cell.get('channel_id'))
print('Protocol:', cell.get('protocol'))

## 3. Per-cycle `summary` — your main table for Week 1–2

In these files, `summary` is a **dictionary of columns**:
- Each key is a column name (e.g. `discharge_capacity`)
- Each value is a **list** with one entry per cycle

We turn that into a pandas DataFrame: **one row per cycle**.

In [ ]:
import pandas as pd

summary = pd.DataFrame(cell['summary'])
print('Shape (rows, columns):', summary.shape)
print('\nColumn names:')
print(list(summary.columns))

In [ ]:
print('First 5 cycles:')
display(summary.head())

print('\nNumeric summary (quick sanity check):')
display(summary.describe())

### What each summary column is (for your dataset description doc)

| Column | Meaning |
|--------|--------|
| `cycle_index` | Cycle number |
| `discharge_capacity` | How much charge delivered on discharge (Ah) |
| `charge_capacity` | Charge capacity (Ah) |
| `dc_internal_resistance` | DC internal resistance |
| `temperature_maximum` / `average` / `minimum` | Temperature stats that cycle |
| `energy_efficiency` | Energy efficiency |
| `charge_duration` | How long charging took |

Copy this table into `docs/week01/dataset_description.md` and adjust if your columns differ on another file.

## 4. Simple plot — capacity fade for this one cell

This is the same kind of curve you will use in Week 2 figures.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(summary['cycle_index'], summary['discharge_capacity'], marker='.', markersize=3)
ax.set_xlabel('Cycle index')
ax.set_ylabel('Discharge capacity (Ah)')
ax.set_title(f"Capacity fade — {cell.get('barcode')}")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. EOL check on this one cell (80% of first cycle capacity)

This matches the logic in `01_data_exploration.ipynb`. We use cycles with `cycle_index >= 1` to skip index 0 if needed.

In [ ]:
s = summary[summary['cycle_index'] >= 1].copy()
initial_cap = s['discharge_capacity'].iloc[0]
threshold = 0.8 * initial_cap

below = s[s['discharge_capacity'] < threshold]
eol = below.iloc[0]['cycle_index'] if len(below) > 0 else s['cycle_index'].iloc[-1]

print(f'Initial capacity (cycle >= 1): {initial_cap:.4f} Ah')
print(f'80% threshold: {threshold:.4f} Ah')
print(f'EOL cycle index: {eol}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(s['cycle_index'], s['discharge_capacity'], label='Discharge capacity')
ax.axhline(threshold, color='red', linestyle='--', label='80% threshold')
ax.axvline(eol, color='orange', linestyle=':', label=f'EOL cycle {eol}')
ax.set_xlabel('Cycle index')
ax.set_ylabel('Discharge capacity (Ah)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. `cycles_interpolated` — voltage / current (large arrays)

These files store **interpolated** time-series data in long format:
- `cycle_index`, `voltage`, `current`, `temperature`, etc. are **parallel lists** of the same length
- Each cycle has many points (often ~2000 per cycle)

You will **not** export all of this in Week 1. You only need to see that it exists for voltage-based features in Week 3.

In [ ]:
curves = cell.get('cycles_interpolated')
if curves is None:
    print('No cycles_interpolated in this file.')
else:
    print('cycles_interpolated columns:')
    for key, value in curves.items():
        if hasattr(value, '__len__') and not isinstance(value, str):
            print(f'  {key:22} length {len(value):,}')
        else:
            print(f'  {key:22} {value}')

In [ ]:
import numpy as np

if curves is not None:
    cycle_to_plot = 10
    idx = np.array(curves['cycle_index']) == cycle_to_plot
    q = np.array(curves['discharge_capacity'])[idx]
    v = np.array(curves['voltage'])[idx]

    print(f'Points in cycle {cycle_to_plot}:', idx.sum())

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(q, v, linewidth=1)
    ax.set_xlabel('Discharge capacity (Ah)')
    ax.set_ylabel('Voltage (V)')
    ax.set_title(f'Discharge curve — cycle {cycle_to_plot}')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 7. Optional — compare two cycles (early vs late)

Severson-style work often uses **voltage curve differences** between early and late cycles (e.g. cycle 10 vs 100). That is Week 3 feature engineering; this is just a preview.

In [ ]:
if curves is not None:
    def discharge_curve(df_cycle_index):
        mask = np.array(curves['cycle_index']) == df_cycle_index
        q = np.array(curves['discharge_capacity'])[mask]
        v = np.array(curves['voltage'])[mask]
        return q, v

    q10, v10 = discharge_curve(10)
    q100, v100 = discharge_curve(100)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(q10, v10, label='Cycle 10')
    ax.plot(q100, v100, label='Cycle 100', alpha=0.8)
    ax.set_xlabel('Discharge capacity (Ah)')
    ax.set_ylabel('Voltage (V)')
    ax.legend()
    ax.set_title('Early vs later discharge voltage curve')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 8. Takeaways — what to do next

1. Fill in `docs/week01/dataset_description.md` using what you printed above.
2. Next notebook (`03_build_cycle_summary.ipynb`): loop **all** files and save `data/processed/cycle_summary.csv` (one row per cell per cycle).
3. Keep `data/processed/cell_targets.csv` as one row per cell with `EOL` and `initial_capacity`.
4. Do **not** put full `cycles_interpolated` into a CSV yet — it is too large; extract features later.

**Week 1 deliverables:** literature summary + dataset description + problem statement (in `docs/week01/`).